# Uplift Modeling - L'Oreal India Example

This notebook demonstrates uplift modeling to identify which customers will respond to discount offers.

**Goal**: Find PERSUADABLES - customers who buy BECAUSE of the discount.

**Avoid**: 
- SURE THINGS - would buy anyway (wasted discount)
- SLEEPING DOGS - discount drives them away (negative effect)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')

## 1. Understanding Customer Segments

| Segment | Without Discount | With Discount | Uplift | Action |
|---------|-----------------|---------------|--------|--------|
| Sure Things | Buy | Buy | 0 | Don't target |
| Persuadables | Don't Buy | Buy | + | TARGET |
| Lost Causes | Don't Buy | Don't Buy | 0 | Don't target |
| Sleeping Dogs | Buy | Don't Buy | - | NEVER target |

In [ ]:
def generate_customer_data(n_customers=10000, seed=42):
    """
    Generate synthetic customer data with heterogeneous treatment effects.
    """
    np.random.seed(seed)
    
    # Customer features
    age = np.random.normal(35, 10, n_customers).clip(18, 65)
    income = np.random.exponential(50000, n_customers) + 20000
    past_purchases = np.random.poisson(3, n_customers)
    website_visits = np.random.poisson(5, n_customers)
    is_loyalty_member = np.random.binomial(1, 0.3, n_customers)
    
    # Baseline purchase probability (without discount)
    baseline_prob = (
        0.05
        + 0.15 * (past_purchases / 10)
        + 0.10 * is_loyalty_member
        + 0.05 * (website_visits / 10)
    ).clip(0.02, 0.6)
    
    # TRUE UPLIFT (what we want to estimate)
    # Different customer segments have different responses
    true_uplift = np.where(
        (income < 60000) & (past_purchases < 3),  # Persuadables
        0.20,
        np.where(
            (income > 80000) & (past_purchases > 5),  # Sure Things
            0.02,
            np.where(
                (age > 50) & (income > 90000),  # Sleeping Dogs
                -0.05,
                0.08  # Average customers
            )
        )
    )
    
    # Random treatment assignment (50/50 split)
    treatment = np.random.binomial(1, 0.5, n_customers)
    
    # Generate outcomes
    prob_with_treatment = (baseline_prob + true_uplift * treatment).clip(0, 1)
    purchased = np.random.binomial(1, prob_with_treatment)
    
    return pd.DataFrame({
        'customer_id': range(n_customers),
        'age': age,
        'income': income,
        'past_purchases': past_purchases,
        'website_visits': website_visits,
        'is_loyalty_member': is_loyalty_member,
        'treatment': treatment,
        'purchased': purchased,
        'true_uplift': true_uplift,
        'baseline_prob': baseline_prob
    })

df = generate_customer_data()

print("Customer Data Summary")
print("=" * 50)
print(f"Total customers: {len(df):,}")
print(f"Treatment group (received discount): {df['treatment'].sum():,}")
print(f"Control group (no discount): {len(df) - df['treatment'].sum():,}")
df.head()

## 2. Overall Conversion Analysis

In [ ]:
# Overall conversion rates
treated = df[df['treatment'] == 1]
control = df[df['treatment'] == 0]

treated_rate = treated['purchased'].mean()
control_rate = control['purchased'].mean()
overall_lift = treated_rate - control_rate

print("Overall Conversion Rates")
print("=" * 50)
print(f"\nTreatment (with discount): {treated_rate:.2%}")
print(f"Control (no discount): {control_rate:.2%}")
print(f"Overall lift: {overall_lift:.2%}")

# Visualize
fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(['Control\n(No Discount)', 'Treatment\n(With Discount)'], 
               [control_rate*100, treated_rate*100], color=['#3498db', '#2ecc71'])
ax.set_ylabel('Conversion Rate (%)')
ax.set_title('Overall Conversion: Control vs Treatment')
for bar, rate in zip(bars, [control_rate, treated_rate]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, 
            f'{rate:.1%}', ha='center', fontsize=12)
plt.tight_layout()
plt.show()

## 3. Train Uplift Model (T-Learner)

**T-Learner** approach:
1. Train Model 1: P(Purchase | Features) for TREATED group
2. Train Model 2: P(Purchase | Features) for CONTROL group
3. Uplift = Model1(X) - Model2(X)

In [ ]:
def train_uplift_model(df):
    """Train T-Learner uplift model."""
    features = ['age', 'income', 'past_purchases', 'website_visits', 'is_loyalty_member']
    X = df[features]
    y = df['purchased']
    treatment = df['treatment']
    
    # Split by treatment
    X_treated = X[treatment == 1]
    y_treated = y[treatment == 1]
    X_control = X[treatment == 0]
    y_control = y[treatment == 0]
    
    # Train two models
    model_treated = RandomForestClassifier(n_estimators=100, random_state=42)
    model_treated.fit(X_treated, y_treated)
    
    model_control = RandomForestClassifier(n_estimators=100, random_state=42)
    model_control.fit(X_control, y_control)
    
    # Predict uplift for all
    prob_treated = model_treated.predict_proba(X)[:, 1]
    prob_control = model_control.predict_proba(X)[:, 1]
    
    uplift_scores = prob_treated - prob_control
    
    return uplift_scores, model_treated, model_control

print("Training T-Learner Uplift Model...")
uplift_scores, model_t, model_c = train_uplift_model(df)
df['uplift_score'] = uplift_scores

print("\nUplift Score Distribution")
print("=" * 50)
print(df['uplift_score'].describe())

In [ ]:
# Visualize uplift score distribution
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(df['uplift_score'], bins=50, edgecolor='black', alpha=0.7)
ax.axvline(x=0, color='red', linestyle='--', label='Zero Uplift')
ax.axvline(x=df['uplift_score'].mean(), color='green', linestyle='--', 
           label=f'Mean ({df["uplift_score"].mean():.3f})')
ax.set_xlabel('Predicted Uplift Score')
ax.set_ylabel('Number of Customers')
ax.set_title('Distribution of Predicted Uplift Scores')
ax.legend()
plt.tight_layout()
plt.show()

## 4. Evaluate Model by Decile

In [ ]:
def evaluate_by_decile(df):
    """Evaluate uplift model by decile."""
    df = df.copy()
    df['decile'] = pd.qcut(df['uplift_score'], 10, labels=False, duplicates='drop') + 1
    
    results = []
    for decile in sorted(df['decile'].unique()):
        decile_df = df[df['decile'] == decile]
        
        treated = decile_df[decile_df['treatment'] == 1]
        control = decile_df[decile_df['treatment'] == 0]
        
        if len(treated) > 0 and len(control) > 0:
            results.append({
                'Decile': decile,
                'N': len(decile_df),
                'Treated Rate': treated['purchased'].mean(),
                'Control Rate': control['purchased'].mean(),
                'Observed Uplift': treated['purchased'].mean() - control['purchased'].mean(),
                'Avg True Uplift': decile_df['true_uplift'].mean(),
                'Avg Score': decile_df['uplift_score'].mean()
            })
    
    return pd.DataFrame(results)

decile_results = evaluate_by_decile(df)

print("Uplift by Decile (Decile 10 = Highest Predicted Uplift)")
print("=" * 80)
print(decile_results.round(3).to_string(index=False))

In [ ]:
# Visualize uplift by decile
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Observed uplift by decile
colors = ['#e74c3c' if u < 0 else '#2ecc71' for u in decile_results['Observed Uplift']]
axes[0].bar(decile_results['Decile'], decile_results['Observed Uplift'] * 100, color=colors)
axes[0].axhline(y=0, color='black', linestyle='-', linewidth=0.5)
axes[0].set_xlabel('Decile (1=Lowest, 10=Highest Predicted Uplift)')
axes[0].set_ylabel('Observed Uplift (percentage points)')
axes[0].set_title('Observed Uplift by Decile')
axes[0].set_xticks(range(1, 11))

# Conversion rates by decile
x = np.arange(len(decile_results))
width = 0.35
axes[1].bar(x - width/2, decile_results['Control Rate'] * 100, width, 
            label='Control', color='#3498db')
axes[1].bar(x + width/2, decile_results['Treated Rate'] * 100, width, 
            label='Treatment', color='#2ecc71')
axes[1].set_xlabel('Decile')
axes[1].set_ylabel('Conversion Rate (%)')
axes[1].set_title('Conversion Rates by Decile')
axes[1].set_xticks(x)
axes[1].set_xticklabels(decile_results['Decile'])
axes[1].legend()

plt.tight_layout()
plt.show()

## 5. Compare Targeting Strategies

In [ ]:
def compare_strategies(df, discount_cost=100, avg_order_value=1500):
    """Compare different targeting strategies."""
    strategies = []
    
    # Strategy 1: Random 50%
    random_incr = df['true_uplift'].mean() * len(df) * 0.5
    random_cost = len(df) * 0.5 * discount_cost
    random_rev = random_incr * avg_order_value
    strategies.append({
        'Strategy': 'Random 50%',
        'Customers': len(df) // 2,
        'Incremental Conv': int(random_incr),
        'Cost (Rs)': int(random_cost),
        'Revenue (Rs)': int(random_rev),
        'ROI': (random_rev - random_cost) / random_cost
    })
    
    # Strategy 2: Top 50% by uplift
    top50 = df[df['uplift_score'] >= df['uplift_score'].quantile(0.5)]
    top50_incr = top50['true_uplift'].sum()
    top50_cost = len(top50) * discount_cost
    top50_rev = top50_incr * avg_order_value
    strategies.append({
        'Strategy': 'Top 50% by Uplift',
        'Customers': len(top50),
        'Incremental Conv': int(top50_incr),
        'Cost (Rs)': int(top50_cost),
        'Revenue (Rs)': int(top50_rev),
        'ROI': (top50_rev - top50_cost) / top50_cost
    })
    
    # Strategy 3: Top 20% by uplift
    top20 = df[df['uplift_score'] >= df['uplift_score'].quantile(0.8)]
    top20_incr = top20['true_uplift'].sum()
    top20_cost = len(top20) * discount_cost
    top20_rev = top20_incr * avg_order_value
    strategies.append({
        'Strategy': 'Top 20% by Uplift',
        'Customers': len(top20),
        'Incremental Conv': int(top20_incr),
        'Cost (Rs)': int(top20_cost),
        'Revenue (Rs)': int(top20_rev),
        'ROI': (top20_rev - top20_cost) / top20_cost
    })
    
    # Strategy 4: Top 50% by BASELINE (wrong approach)
    baseline50 = df[df['baseline_prob'] >= df['baseline_prob'].quantile(0.5)]
    baseline50_incr = baseline50['true_uplift'].sum()
    baseline50_cost = len(baseline50) * discount_cost
    baseline50_rev = baseline50_incr * avg_order_value
    strategies.append({
        'Strategy': 'Top 50% by Baseline (WRONG)',
        'Customers': len(baseline50),
        'Incremental Conv': int(baseline50_incr),
        'Cost (Rs)': int(baseline50_cost),
        'Revenue (Rs)': int(baseline50_rev),
        'ROI': (baseline50_rev - baseline50_cost) / baseline50_cost
    })
    
    return pd.DataFrame(strategies)

roi_comparison = compare_strategies(df)

print("Targeting Strategy Comparison")
print("=" * 90)
print(roi_comparison.to_string(index=False))

In [ ]:
# Visualize ROI comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROI by strategy
colors = ['#95a5a6', '#2ecc71', '#27ae60', '#e74c3c']
axes[0].barh(roi_comparison['Strategy'], roi_comparison['ROI'] * 100, color=colors)
axes[0].set_xlabel('ROI (%)')
axes[0].set_title('ROI by Targeting Strategy')
axes[0].axvline(x=0, color='black', linestyle='-', linewidth=0.5)

# Incremental conversions
axes[1].barh(roi_comparison['Strategy'], roi_comparison['Incremental Conv'], color=colors)
axes[1].set_xlabel('Incremental Conversions')
axes[1].set_title('Incremental Conversions by Strategy')

plt.tight_layout()
plt.show()

print("\nKey Insight: Targeting by UPLIFT has higher ROI than targeting by baseline probability!")

## 6. Identify Customer Segments

In [ ]:
# Segment customers
df['segment'] = np.where(
    (df['baseline_prob'] > 0.3) & (df['true_uplift'] < 0.05),
    'Sure Things',
    np.where(
        (df['baseline_prob'] < 0.15) & (df['true_uplift'] > 0.10),
        'Persuadables',
        np.where(
            df['true_uplift'] < 0,
            'Sleeping Dogs',
            np.where(
                (df['baseline_prob'] < 0.10) & (df['true_uplift'] < 0.05),
                'Lost Causes',
                'Average'
            )
        )
    )
)

segment_summary = df.groupby('segment').agg({
    'customer_id': 'count',
    'baseline_prob': 'mean',
    'true_uplift': 'mean',
    'uplift_score': 'mean'
}).round(3)
segment_summary.columns = ['Count', 'Avg Baseline', 'Avg True Uplift', 'Avg Predicted Uplift']

print("Customer Segments")
print("=" * 70)
print(segment_summary)

# Visualize
fig, ax = plt.subplots(figsize=(10, 6))
colors = {'Sure Things': '#3498db', 'Persuadables': '#2ecc71', 
          'Lost Causes': '#95a5a6', 'Sleeping Dogs': '#e74c3c', 'Average': '#f39c12'}

for segment in df['segment'].unique():
    segment_data = df[df['segment'] == segment]
    ax.scatter(segment_data['baseline_prob'], segment_data['true_uplift'], 
               alpha=0.5, label=segment, color=colors.get(segment, 'gray'))

ax.axhline(y=0, color='black', linestyle='--', alpha=0.5)
ax.set_xlabel('Baseline Purchase Probability')
ax.set_ylabel('True Uplift')
ax.set_title('Customer Segmentation by Baseline Probability and Uplift')
ax.legend()
plt.tight_layout()
plt.show()

## Key Takeaways

1. **Uplift = P(buy|discount) - P(buy|no discount)** - not just who buys
2. **Persuadables are the target** - they change behavior because of treatment
3. **Sure Things waste budget** - they would buy anyway
4. **Sleeping Dogs are dangerous** - treatment drives them away
5. **T-Learner** trains separate models for treatment and control
6. **Evaluate by decile** - top deciles should have highest uplift
7. **Uplift targeting has higher ROI** than targeting by baseline probability
8. **Requires randomized data** - need treatment and control groups